In [ ]:
"""
Feature Reduction — ProtT5 Embeddings (1024-dim)
"""


'\nFeature Reduction — ProtT5 Embeddings (1024-dim)\n================================================================\nGoal: Identify reduced feature subsets that retain (or improve) the\n76% baseline accuracy achieved with the full 1024-dim ProtT5 embedding.\n\nMethods retained (5):\n  1. L1 (Lasso)         — sparse linear selection, fast on dense embeddings\n  2. Random Forest Imp.  — robust, captures non-linear importance\n  3. RFE                 — iterative elimination, adaptive target size\n  4. Correlation Filter  — removes near-duplicate dims (threshold raised to 0.95)\n  5. Mutual Information  — captures non-linear feature-label dependence\n\nMethods REMOVED:\n  - Boruta : too slow on 1024-dim × 7170 rows (hours on CPU), marginal\n             benefit over RF importance which is already included.\n  - L2/Ridge : does not produce a sparse subset — "top 80% by |coef|"\n             on Ridge retains ~819/1024 features, barely a reduction\n             and redundant with the no-se

In [2]:
import pandas as pd
import numpy as np
import warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import RFE, mutual_info_classif, VarianceThreshold

warnings.filterwarnings('ignore')

In [ ]:
# ── PATHS ─────────────────────────────────────────────────────────────────────
FEATURES_PATH = '/kaggle/input/datasets/hsharmaa/plm-food-6356/features_prott5_6356.csv'      
LABELS_PATH   = '/kaggle/input/datasets/hsharmaa/food-6356/Food_6356.csv'                  
LABEL_COL_IDX = 1                                
PLM_TAG       = 'prott5'                         

# ── LOAD ──────────────────────────────────────────────────────────────────────

In [4]:
# ── LOAD ──────────────────────────────────────────────────────────────────────
print("Loading data...")
X = pd.read_csv(FEATURES_PATH)
df_labels = pd.read_csv(LABELS_PATH)
y_raw = df_labels[df_labels.columns[LABEL_COL_IDX]]
y = pd.Series(LabelEncoder().fit_transform(y_raw.values))  # FIX: ensure 0/1/2 encoding

ORIGINAL_DIM = X.shape[1]
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Classes: {sorted(y.unique())} | Counts:\n{y.value_counts()}\n")

assert X.shape[0] == len(y), f"Row mismatch: X={X.shape[0]}, y={len(y)}"

results = {}


Loading data...
X shape: (6356, 1024), y shape: (6356,)
Classes: [np.int64(0), np.int64(1), np.int64(2)] | Counts:
0    2810
2    1980
1    1566
Name: count, dtype: int64



In [ ]:
# ============================================================
# 1. L1 — LASSO
# ============================================================
print("=" * 55)
print("1. L1 Lasso Regression (Multiclass)")
print("=" * 55)
scaler_l1 = StandardScaler()
X_scaled_l1 = scaler_l1.fit_transform(X)
l1_model = LogisticRegression(solver='saga', penalty='elasticnet', l1_ratio=1.0,
                               C=0.01, random_state=42, max_iter=3000, n_jobs=-1)
l1_model.fit(X_scaled_l1, y)
l1_coef = np.abs(l1_model.coef_).max(axis=0)
l1_support = l1_coef > 0
X.loc[:, l1_support].to_csv(f'{PLM_TAG}_L1.csv', index=False)
results['L1'] = int(l1_support.sum())
print(f"L1 selected {l1_support.sum()} features -> {PLM_TAG}_L1.csv\n")

# ============================================================
# 2. RANDOM FOREST IMPORTANCE
# ============================================================
print("=" * 55)
print("2. Random Forest Feature Importance")
print("=" * 55)
rf = RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=42,
                             class_weight='balanced', max_features=0.3)
rf.fit(X, y)
importances = pd.Series(rf.feature_importances_, index=X.columns)
rf_support = importances >= np.percentile(importances, 80)
X.loc[:, rf_support].to_csv(f'{PLM_TAG}_RF.csv', index=False)
results['Random Forest'] = int(rf_support.sum())
print(f"RF selected {rf_support.sum()} features -> {PLM_TAG}_RF.csv\n")

# ============================================================
# 3. RFE
# ============================================================
print("=" * 55)
print("3. Recursive Feature Elimination (RFE)")
print("=" * 55)
scaler_rfe = StandardScaler()
X_scaled_rfe = scaler_rfe.fit_transform(X)
rfe_estimator = LogisticRegression(solver='saga', penalty='elasticnet', l1_ratio=1.0,
                                    C=0.1, random_state=42, max_iter=3000, n_jobs=-1)

n_target = max(50, ORIGINAL_DIM // 4)
rfe = RFE(estimator=rfe_estimator, n_features_to_select=n_target, step=20, verbose=1)
rfe.fit(X_scaled_rfe, y)
X.loc[:, rfe.support_].to_csv(f'{PLM_TAG}_RFE.csv', index=False)
results['RFE'] = int(rfe.support_.sum())
print(f"RFE selected {rfe.support_.sum()} features (target={n_target}) -> {PLM_TAG}_RFE.csv\n")


1. L1 Lasso Regression (Multiclass)
L1 selected 148 features -> prott5_L1.csv

2. Random Forest Feature Importance
RF selected 205 features -> prott5_RF.csv

3. Recursive Feature Elimination (RFE)
Fitting estimator with 1024 features.
Fitting estimator with 1004 features.
Fitting estimator with 984 features.
Fitting estimator with 964 features.
Fitting estimator with 944 features.
Fitting estimator with 924 features.
Fitting estimator with 904 features.
Fitting estimator with 884 features.
Fitting estimator with 864 features.
Fitting estimator with 844 features.
Fitting estimator with 824 features.
Fitting estimator with 804 features.
Fitting estimator with 784 features.
Fitting estimator with 764 features.
Fitting estimator with 744 features.
Fitting estimator with 724 features.
Fitting estimator with 704 features.
Fitting estimator with 684 features.
Fitting estimator with 664 features.
Fitting estimator with 644 features.
Fitting estimator with 624 features.
Fitting estimator with 6

In [ ]:
# ============================================================
# 4. CORRELATION FILTER
# ============================================================
print("=" * 55)
print("4. Correlation Filter")
print("=" * 55)

X_var = X.copy()
print(f"Features going into correlation filter: {X_var.shape[1]}")

corr_matrix = X_var.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Diagnostic: see the actual correlation distribution before picking threshold
upper_vals = upper_tri.values[~np.isnan(upper_tri.values)]
print(f"Pairwise |correlation| stats: "
      f"max={upper_vals.max():.3f}, "
      f"95th pct={np.percentile(upper_vals,95):.3f}, "
      f"99th pct={np.percentile(upper_vals,99):.3f}")

CORR_THRESHOLD = 0.85
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > CORR_THRESHOLD)]
X_corr = X_var.drop(columns=to_drop)
X_corr.to_csv(f'{PLM_TAG}_Correlation.csv', index=False)
results['Correlation'] = X_corr.shape[1]
print(f"Correlation (>{CORR_THRESHOLD}) removed {len(to_drop)} features, "
      f"kept {X_corr.shape[1]} -> {PLM_TAG}_Correlation.csv\n")

4. Correlation Filter
Features going into correlation filter: 1024
Pairwise |correlation| stats: max=0.867, 95th pct=0.257, 99th pct=0.343
Correlation (>0.85) removed 1 features, kept 1023 -> prott5_Correlation.csv



In [7]:
# ============================================================
# 5. MUTUAL INFORMATION
# ============================================================
print("=" * 55)
print("5. Mutual Information")
print("=" * 55)
mi_scores = mutual_info_classif(X, y, random_state=42, n_neighbors=5)
mi_series = pd.Series(mi_scores, index=X.columns)
mi_support = mi_series >= np.percentile(mi_series, 80)
X.loc[:, mi_support].to_csv(f'{PLM_TAG}_MI.csv', index=False)
results['Mutual Information'] = int(mi_support.sum())
print(f"MI selected {mi_support.sum()} features -> {PLM_TAG}_MI.csv\n")


5. Mutual Information
MI selected 205 features -> prott5_MI.csv



In [8]:
# ============================================================
# 6. PCA (Dimensionality Reduction, not Selection)
# ============================================================
print("=" * 55)
print("6. PCA (95% Variance Retention)")
print("=" * 55)
from sklearn.decomposition import PCA

scaler_pca = StandardScaler()
X_scaled_pca = scaler_pca.fit_transform(X)

pca = PCA(n_components=0.95, random_state=42)   # retain 95% variance
X_pca = pca.fit_transform(X_scaled_pca)

pca_cols = [f'pc_{i}' for i in range(X_pca.shape[1])]
pd.DataFrame(X_pca, columns=pca_cols).to_csv(f'{PLM_TAG}_PCA.csv', index=False)
results['PCA (95% var)'] = X_pca.shape[1]
print(f"PCA retained {X_pca.shape[1]} components (95% variance) -> {PLM_TAG}_PCA.csv\n")

6. PCA (95% Variance Retention)
PCA retained 391 components (95% variance) -> prott5_PCA.csv



In [9]:
# ============================================================
# SUMMARY
# ============================================================
print("=" * 55)
print("FINAL SUMMARY")
print("=" * 55)
print(f"{'Method':<22} {'Features Selected':>17} {'% of ' + str(ORIGINAL_DIM):>12}")
print("-" * 55)
for method, count in results.items():
    print(f"{method:<22} {count:>17} {count/ORIGINAL_DIM*100:>11.1f}%")
print("-" * 55)
print(f"{'Original (baseline)':<22} {ORIGINAL_DIM:>17} {'100.0':>11}%")
print("\nAll files saved successfully!")
print("\nNEXT STEP: Run each {PLM_TAG}_<method>.csv through the 5-fold CV")
print("classification script and compare Test_ACC/F1/MCC against the")
print(f"76% baseline from the full {ORIGINAL_DIM}-dim ProtT5 embedding.")

FINAL SUMMARY
Method                 Features Selected    % of 1024
-------------------------------------------------------
L1                                   148        14.5%
Random Forest                        205        20.0%
RFE                                  256        25.0%
Correlation                         1023        99.9%
Mutual Information                   205        20.0%
PCA (95% var)                        391        38.2%
-------------------------------------------------------
Original (baseline)                 1024       100.0%

All files saved successfully!

NEXT STEP: Run each {PLM_TAG}_<method>.csv through the 5-fold CV
classification script and compare Test_ACC/F1/MCC against the
76% baseline from the full 1024-dim ProtT5 embedding.
